# Stage 0 pilot runner — Colab

Lightweight wrapper around `train.py`. Outputs persist to Drive so a session disconnect does not lose work.

Designed to work across three front-ends with the same notebook:
- **Native Colab (browser):** the clone cell fetches the repo into `/content/repo`.
- **VS Code + Colab extension:** the local workspace is already synced; the clone cell becomes a no-op.
- **Local Jupyter / VS Code Jupyter (no Colab):** runs against your local Python; uses the repo on disk; outputs go to `/tmp`. Useful for dry-runs and notebook development. The training cells will use CPU and be slow.

## Before running
1. Pro tier minimum is recommended for actual training (free tier idle disconnects make the suite painful).
2. The setup cell prints the allocated GPU type and measures throughput — use those numbers, not the project plan's GPU assumptions, to budget.
3. The suite is **idempotent**: re-running the round cell skips runs whose `eval.json` already exists, so reconnect after a disconnect and just re-run.

In [ ]:
# 1) Mount Drive + define suite-wide variables (re-run after every reconnect)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/concept_critic'
except ImportError:
    print('Not running on Colab — using local /tmp for outputs')
    DRIVE_ROOT = '/tmp/concept_critic'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)

# Suite-wide variables. Used by every round cell. Edit here if you need to override.
OUTPUT_DIR  = f'{DRIVE_ROOT}/stage0'
BENCHMARKS  = 'armed_corridor phase_crossing'
MAX_MINUTES = 660   # ~11h, safely under Colab Pro session cap

print('output root:', DRIVE_ROOT)
print('OUTPUT_DIR :', OUTPUT_DIR)
print('BENCHMARKS :', BENCHMARKS)
print('MAX_MINUTES:', MAX_MINUTES)

In [ ]:
# 2) Locate or fetch the repo. Handles three cases:
#    - Native Colab: clone into /content/repo if not already present
#    - VS Code + Colab extension: workspace already synced, train.py is in cwd or an ancestor
#    - Local Jupyter (no Colab): same as above — train.py is somewhere up the tree from the notebook
import os, subprocess

REPO_URL = 'https://github.com/AdeX11/concept_critic_models.git'
BRANCH   = 'domingo-experimental'   # pin a commit SHA for reproducibility, e.g. 'b70d8ac'

def _find_repo_root() -> str | None:
    cur = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(cur, 'train.py')):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    if os.path.exists('/content/repo/train.py'):
        return '/content/repo'
    return None

REPO_DIR = _find_repo_root()
if REPO_DIR is None:
    if not os.path.isdir('/content'):
        raise RuntimeError(
            'train.py not found in cwd ancestors and /content does not exist. '
            'Open this notebook from inside the cloned repo, or run it on a Colab runtime.'
        )
    REPO_DIR = '/content/repo'
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR])

os.chdir(REPO_DIR)
print('repo  :', REPO_DIR)
print('head  :', subprocess.check_output(['git', 'rev-parse', 'HEAD']).decode().strip())
print('branch:', subprocess.check_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']).decode().strip())

In [ ]:
# 3) Install deps + smoke + measure throughput on the GPU you actually got
!bash colab/setup.sh

In [ ]:
# 4) Round 1 — temporal architecture pilot (no prior results needed)
#    20 runs (10 configs × 2 benchmarks) × 300k timesteps each
!python colab/run_suite.py \
    --round round1 \
    --benchmarks $BENCHMARKS \
    --output_dir $OUTPUT_DIR \
    --max_minutes $MAX_MINUTES

## After Round 1 — aggregate and sanity-check before Round 2

Look at the printed CSV. Sanity checks before proceeding:
- 20 rows present (10 configs × 2 benchmarks)
- Some configs have non-zero `success_rate`
- On `armed_corridor` (hidden), `gru` should beat `none` — if not, pause and investigate, do not proceed.

In [ ]:
!python cluster/aggregate.py --results_root $OUTPUT_DIR --csv_out $OUTPUT_DIR/round1_aggregate.csv
!head -25 $OUTPUT_DIR/round1_aggregate.csv

## Round 2 — LR × ent_coef pilot

24 runs × 300k. Depends on Round 1 results (winners are picked automatically by `cluster/run_pilot.py`).
Re-running this cell after a disconnect skips completed runs.

In [ ]:
!python colab/run_suite.py \
    --round round2 \
    --benchmarks $BENCHMARKS \
    --results_root $OUTPUT_DIR \
    --output_dir   $OUTPUT_DIR \
    --max_minutes  $MAX_MINUTES

## Round 3 — λ_v × λ_s pilot (CAC only)

8 runs × 300k. Depends on Round 2 results.
Independent of Round 4 — they can run in either order.

In [ ]:
!python colab/run_suite.py \
    --round round3 \
    --benchmarks $BENCHMARKS \
    --results_root $OUTPUT_DIR \
    --output_dir   $OUTPUT_DIR \
    --max_minutes  $MAX_MINUTES

## Round 4 — vanilla_freeze label budget pilot

12 runs × 300k. Depends on Round 2 results.

In [ ]:
!python colab/run_suite.py \
    --round round4 \
    --benchmarks $BENCHMARKS \
    --results_root $OUTPUT_DIR \
    --output_dir   $OUTPUT_DIR \
    --max_minutes  $MAX_MINUTES

## 3-seed revalidation

Run only after Rounds 1–4 are all complete. 18 runs × 300k.
Re-runs each method's best config across seeds 42, 123, 456 to confirm winners aren't single-seed noise.

In [ ]:
!python colab/run_suite.py \
    --round revalidate \
    --benchmarks $BENCHMARKS \
    --results_root $OUTPUT_DIR \
    --output_dir   $OUTPUT_DIR \
    --max_minutes  $MAX_MINUTES

## Long-horizon confirmation — switch to A100 first

Each run is 1M timesteps. On L4 that's ~10 min/run; on A100 ~3 min/run.

**Before running this cell:**
1. Runtime → Change runtime type → **A100 GPU**. Save, accept restart.
2. Re-run cells 1, 2, 3 (Drive mount, repo locate, setup) — the runtime VM was reset.
3. Then run the cell below.

6 runs × 1M = ~20–30 min on A100.

In [ ]:
!python colab/run_suite.py \
    --round confirm \
    --benchmarks $BENCHMARKS \
    --results_root $OUTPUT_DIR \
    --output_dir   $OUTPUT_DIR \
    --max_minutes  $MAX_MINUTES

## CAC pre-fix architecture ablation

Single 1M run on `armed_corridor` (hidden) only. ~3–5 min on A100.

In [ ]:
!python colab/run_suite.py \
    --round ablation \
    --results_root $OUTPUT_DIR \
    --output_dir   $OUTPUT_DIR \
    --max_minutes  $MAX_MINUTES

## Final aggregation

Produce one CSV with all completed runs and confirm the total count.

In [ ]:
!python cluster/aggregate.py --results_root $OUTPUT_DIR --csv_out $OUTPUT_DIR/stage0_full.csv
import pandas as pd
df = pd.read_csv(f'{OUTPUT_DIR}/stage0_full.csv')
print('total runs:', len(df))
print()
print(df.groupby(['method', 'benchmark_id']).size().rename('count'))

## Monitoring & troubleshooting

**Live progress while a round is running** — open a second cell below and run:

```python
!tail -10 $OUTPUT_DIR/_suite_progress.jsonl
```

**TensorBoard for a specific run:**

```python
%load_ext tensorboard
%tensorboard --logdir $OUTPUT_DIR
```

**Inspecting a failed run** (look for `FAIL(rc=…)` in the suite output):

```python
!cat $OUTPUT_DIR/<run_dir_name>/train.log | tail -40
```

**Disconnect recovery:** reconnect → re-run cells 1, 2, 3 → re-run whichever round cell was active. Completed runs are skipped via the `eval.json` presence check.